In [ ]:

import os, time, math
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.lines import Line2D

OUT = "figures"
os.makedirs(OUT, exist_ok=True)
T0 = time.time()


THETA = (0.5, 1.5)                      # range of kappa
KG = np.linspace(THETA[0], THETA[1], 2001)
ETA = 0.95                              # safety threshold: P(no drop) >= ETA
P_DROP_MAX = 1.0 - ETA
BDEG = 60                               # degree of the Bernstein polynomials

TYPES = ["box", "crate", "casting"]
# kappa threshold above which lifting a part of mass m next to the operator is safe
KC_TARGET = {
    "box":     dict(zip(range(1, 9),  [0.40, 0.48, 0.56, 0.64, 0.72, 0.80, 0.88, 0.96])),
    "crate":   dict(zip(range(3, 11), [0.55, 0.62, 0.69, 0.76, 0.83, 0.90, 0.95, 1.06])),
    "casting": dict(zip(range(8, 15), [0.76, 0.84, 0.97, 1.00, 1.03, 1.06, 1.09])),
}
FIXTURE_CAP = 6                         # kg: partial lifts on the fixture only up to this mass
PROBE_TYPES = ("box", "crate")          # castings do not fit the fixture
QUERY_P = {"box": 0.45, "crate": 0.35, "casting": 0.20}

def _bernstein_basis(n, x):
    t = (x - THETA[0]) / (THETA[1] - THETA[0])
    k = np.arange(n + 1)
    C = np.array([math.comb(n, j) for j in k], dtype=float)
    return C[None, :] * t[:, None] ** k[None, :] * (1 - t[:, None]) ** (n - k[None, :])

BASIS = _bernstein_basis(BDEG, KG)
NODES = np.linspace(THETA[0], THETA[1], BDEG + 1)

def bernstein_sigmoid(loc, tau):
    """Bernstein polynomial (degree BDEG) of p(kappa) = sigmoid((loc - kappa)/tau)."""
    vals = 1.0 / (1.0 + np.exp(-(loc - NODES) / tau))
    return BASIS @ vals

def crossing(f, level):
    """Smallest kappa on the grid with f(kappa) <= level (f decreasing)."""
    idx = np.searchsorted(-f, -level)
    return KG[min(idx, len(KG) - 1)] if idx < len(KG) else THETA[1] + 1.0

# drop probability f (safety) and micro-slip probability g (tactile signal used for learning).
# F, Gm: curves of the machine's model.  FT, GT: curves of the true environment.
F, Gm, KC, FT, GT = {}, {}, {}, {}, {}

def build_default():
    """Bernstein curves (degree BDEG) calibrated on KC_TARGET; the true world uses the same curves."""
    for G in TYPES:
        for m, kc in KC_TARGET[G].items():
            if THETA[0] + 0.05 <= kc <= THETA[1] - 0.05:
                lo, hi = kc - 0.6, kc + 0.6
                for _ in range(60):
                    mid = 0.5 * (lo + hi)
                    if crossing(bernstein_sigmoid(mid, 0.03), P_DROP_MAX) < kc: lo = mid
                    else: hi = mid
                f = bernstein_sigmoid(0.5 * (lo + hi), 0.03)
            else:
                f = bernstein_sigmoid(kc - 0.09, 0.03)
            F[(G, m)] = np.clip(f, 1e-6, 1 - 1e-6)
            KC[(G, m)] = crossing(F[(G, m)], P_DROP_MAX)
            Gm[(G, m)] = np.clip(np.maximum(bernstein_sigmoid(kc + 0.08, 0.06), F[(G, m)]), 1e-6, 1 - 1e-6)
            FT[(G, m)], GT[(G, m)] = F[(G, m)], Gm[(G, m)]

def physics_curves(kc, wd=0.03, wg=0.06):
    """Exact logistic physics (widths wd, wg): drop probability equals 1 - ETA exactly at kappa = kc."""
    f = 1.0 / (1.0 + np.exp(-((kc - wd * math.log((1 - P_DROP_MAX) / P_DROP_MAX)) - KG) / wd))
    g = np.maximum(1.0 / (1.0 + np.exp(-((kc + 0.08) - KG) / wg)), f)
    return np.clip(f, 1e-6, 1 - 1e-6), np.clip(g, 1e-6, 1 - 1e-6)

def approximate_raw(y, family):
    if family == "linear":
        z = np.polyval(np.polyfit(KG, y, 1), KG)
    elif family.startswith("chebyshev"):
        z = np.polynomial.chebyshev.Chebyshev.fit(KG, y, int(family.split("-")[1]))(KG)
    elif family.startswith("bernstein"):
        n = int(family.split("-")[1]); nodes = np.linspace(THETA[0], THETA[1], n + 1)
        z = _bernstein_basis(n, KG) @ np.interp(nodes, KG, y)
    return z

def is_valid(z):
    """Probability curve in [0, 1] and nonincreasing in kappa (Assumption 1), up to rounding."""
    return bool((z >= -1e-9).all() and (z <= 1 + 1e-9).all() and (np.diff(z) <= 1e-9).all())

def approximate(y, family):
    return np.clip(approximate_raw(y, family), 1e-6, 1 - 1e-6)

def build_family(family, truth, wd=0.03, wg=0.06):
    """Model curves from a polynomial family, calibrated so that the safety threshold of every part is the
    same as in the physics; truth = 'same' (the world follows the model curves) or 'physics' (exact logistic
    curves). Returns the number of part instances whose approximated curves are not valid probability curves."""
    invalid = 0
    for G in TYPES:
        for m, kc in KC_TARGET[G].items():
            fp, gp = physics_curves(kc, wd, wg)
            if THETA[0] + 0.05 <= kc <= THETA[1] - 0.05:
                lo, hi = kc - 0.6, kc + 0.6
                for _ in range(50):
                    mid = 0.5 * (lo + hi)
                    sig = 1.0 / (1.0 + np.exp(-(mid - KG) / wd))
                    if crossing(approximate(sig, family), P_DROP_MAX) < kc: lo = mid
                    else: hi = mid
                fr = approximate_raw(1.0 / (1.0 + np.exp(-(0.5 * (lo + hi) - KG) / wd)), family)
            else:
                fr = approximate_raw(fp, family)
            gr = approximate_raw(gp, family)
            invalid += int(not (is_valid(fr) and is_valid(gr)))
            f = np.clip(fr, 1e-6, 1 - 1e-6)
            F[(G, m)] = f
            Gm[(G, m)] = np.clip(np.maximum(gr, f), 1e-6, 1 - 1e-6)
            KC[(G, m)] = crossing(F[(G, m)], P_DROP_MAX)
            FT[(G, m)], GT[(G, m)] = (F[(G, m)], Gm[(G, m)]) if truth == "same" else (fp, gp)
    return invalid

build_default()

def _idx(k):
    return int(np.clip(np.searchsorted(KG, k), 0, len(KG) - 1))

CERT = {"mode": "exact", "certs": 0, "unsound": 0}

def permit(G, m, a, b):
    """Safe verdict over [a, b]. In 'monotone' mode the certificate relies on Assumption 1: a nonincreasing drop
    curve attains its maximum at kappa = a, so a single evaluation suffices; unsound certificates are counted."""
    exact_ok = F[(G, m)][_idx(a):_idx(b) + 1].max() <= P_DROP_MAX
    if CERT["mode"] == "monotone":
        ok = F[(G, m)][_idx(a)] <= P_DROP_MAX
        if ok:
            CERT["certs"] += 1; CERT["unsound"] += int(not exact_ok)
        return ok
    return exact_ok

def block(G, m, a, b):
    if CERT["mode"] == "monotone":
        return F[(G, m)][_idx(b)] > P_DROP_MAX
    return F[(G, m)][_idx(a):_idx(b) + 1].min() > P_DROP_MAX

def kl_cat(p, q):
    p = np.clip(p, 1e-12, 1); q = np.clip(q, 1e-12, 1)
    return float(np.sum(p * np.log(p / q)))

def lift_dist(G, m, k):
    return outcome_probs(val(Gm[(G, m)], k), val(F[(G, m)], k))

def ml_estimate(know, G, a, b):
    """Estimate w_hat_k: maximum-likelihood factor of type G within its range [a, b] over W_k."""
    i0, i1 = _idx(a), _idx(b)
    ll = np.zeros(i1 - i0 + 1); seen = False
    for p in PAIRS:
        if p[0] != G or know.n[p] == 0: continue
        g = Gm[p[:2]][i0:i1 + 1]; seen = True
        ll += know.s[p] * np.log(g) + (know.n[p] - know.s[p]) * np.log(1 - g)
    return KG[i0 + int(np.argmax(ll))] if seen else 0.5 * (a + b)

def opposite_world(G, m, a, b, khat):
    """Closest kappa in [a, b] whose verdict differs from the verdict at khat (None if there is none)."""
    seg = F[(G, m)][_idx(a):_idx(b) + 1]
    ok = seg <= P_DROP_MAX; ok_hat = val(F[(G, m)], khat) <= P_DROP_MAX
    idx = np.nonzero(ok != ok_hat)[0]
    if len(idx) == 0: return None
    ks = KG[_idx(a) + idx]
    return ks[np.argmin(np.abs(ks - khat))]

QUERIES = [(G, m) for G in TYPES for m in KC_TARGET[G]]
PAIRS = [(G, m, "lift") for (G, m) in QUERIES] + \
        [(G, m, "probe") for (G, m) in QUERIES if G in PROBE_TYPES and m <= FIXTURE_CAP]
NPAIRS = len(PAIRS)

def val(arr, k):
    return float(np.interp(k, KG, arr))

def true_safe(G, m, kap):
    return val(FT[(G, m)], kap[G]) <= P_DROP_MAX

DELTA = 0.05

def kl_bern(p, q):
    p = min(max(p, 1e-12), 1 - 1e-12); q = min(max(q, 1e-12), 1 - 1e-12)
    return p * math.log(p / q) + (1 - p) * math.log((1 - p) / (1 - q))

def ci_kl(s, n):
    """Anytime Chernoff (KL) interval for a Bernoulli mean, union bound over pairs and n."""
    if n == 0: return 0.0, 1.0
    L = math.log(4 * NPAIRS * n * n / DELTA)
    ph = s / n
    lo, hi = 0.0, ph
    for _ in range(40):
        mid = 0.5 * (lo + hi)
        if n * kl_bern(ph, mid) > L: lo = mid
        else: hi = mid
    plo = lo
    lo, hi = ph, 1.0
    for _ in range(40):
        mid = 0.5 * (lo + hi)
        if n * kl_bern(ph, mid) > L: hi = mid
        else: lo = mid
    return plo, hi

def ci_l1(s, n):
    """l1 ball of eq. (10) for a two-outcome distribution."""
    if n == 0: return 0.0, 1.0
    L = 2 * math.log(2) + math.log(2 * 2 * NPAIRS * n * n / DELTA)
    r = math.sqrt(2 * L / n)
    return max(0.0, s / n - r / 2), min(1.0, s / n + r / 2)

def kappa_interval(g, plo, phi):
    """{kappa : g(kappa) in [plo, phi]}. 'exact': hull of the set; 'monotone': first and last crossing, which
    is exact for a nonincreasing g (Assumption 1)."""
    if CERT["mode"] == "monotone":
        i1 = np.searchsorted(-g, -phi); i2 = np.searchsorted(-g, -plo, side="right") - 1
        if i1 > i2 or i1 >= len(KG): return None
        return KG[i1], KG[max(i2, 0)]
    idx = np.nonzero((g >= plo) & (g <= phi))[0]
    if len(idx) == 0: return None
    return KG[idx[0]], KG[idx[-1]]

class Knowledge:
    """Data D_k, per-pair confidence sets B_k(s,a) and per-unit parameter intervals."""
    def __init__(self, ci="kl"):
        self.n = {p: 0 for p in PAIRS}; self.s = {p: 0 for p in PAIRS}
        self.counts = {p: np.zeros(3, dtype=int) for p in PAIRS}   # D_k: transitions to stable / micro-slip / drop
        self.ci = ci_kl if ci == "kl" else ci_l1
        self.unit = {G: (THETA[0], THETA[1]) for G in TYPES}
        self.pair_int = {}
        self.empty_unit = False
        self.truth, self.cov_fail = None, False

    def add(self, pair, out):
        """Adds the observed successors `out` = (stable, micro-slip, drop) of pair to D_k and updates B_k."""
        self.counts[pair] += out
        self.n[pair] += int(out.sum()); self.s[pair] += int(out[1] + out[2])
        G, m, _ = pair
        plo, phi = self.ci(self.s[pair], self.n[pair])
        self.pair_int[pair] = kappa_interval(Gm[(G, m)], plo, phi)
        lo, hi = THETA
        for p, I in self.pair_int.items():
            if p[0] != G: continue
            if I is None: lo, hi = 1.0, 0.0; break
            lo, hi = max(lo, I[0]), min(hi, I[1])
        self.unit[G] = (lo, hi)
        if self.truth is not None and not (lo - 1e-9 <= self.truth[G] <= hi + 1e-9): self.cov_fail = True

    def ranges(self, eps):
        """Range of each kappa_G over the consistent admissible worlds W_k (None if W_k is empty)."""
        lows = [self.unit[G][0] for G in TYPES]; highs = [self.unit[G][1] for G in TYPES]
        if any(l > h for l, h in zip(lows, highs)): return None
        kb_lo = max(max(lows) - eps, THETA[0]); kb_hi = min(min(highs) + eps, THETA[1])
        if kb_lo > kb_hi: return None
        out = {}
        for G in TYPES:
            a = max(self.unit[G][0], kb_lo - eps, THETA[0]); b = min(self.unit[G][1], kb_hi + eps, THETA[1])
            if a > b: return None
            out[G] = (a, b)
        return out

    def nominal_rejected(self):
        return self.ranges(0.0) is None

from itertools import combinations

def _permit_exact(G, m, a, b):
    return F[(G, m)][_idx(a):_idx(b) + 1].max() <= P_DROP_MAX

def _block_exact(G, m, a, b):
    return F[(G, m)][_idx(a):_idx(b) + 1].min() > P_DROP_MAX

def nominal_ranges(know, untied, eps):
    """Range of each factor over the consistent nominal parameters C_k. Types not in `untied` share one factor
    theta_bar; after a revision (Remark 3) the types in `untied` have their own factor within eps of theta_bar,
    so that the revised class stays inside the admissible class. Returns None when C_k is empty (rejection)."""
    lo, hi = THETA
    for G in TYPES:
        l, h = know.unit[G]
        if l > h: return None
        if G in untied: lo, hi = max(lo, l - eps), min(hi, h + eps)
        else: lo, hi = max(lo, l), min(hi, h)
    if lo > hi: return None
    out = {}
    for G in TYPES:
        l, h = know.unit[G]
        out[G] = (max(l, lo - eps, THETA[0]), min(h, hi + eps, THETA[1])) if G in untied else (lo, hi)
        if out[G][0] > out[G][1]: return None
    return out

def revise(know, untied, eps):
    """Remark 3: gives an independent factor to the smallest set of further types that restores consistency."""
    rest = [G for G in TYPES if G not in untied]
    for k in range(1, len(rest) + 1):
        for U in combinations(rest, k):
            if nominal_ranges(know, set(untied) | set(U), eps) is not None:
                return frozenset(set(untied) | set(U))
    return None

def explain_model_dependent(know, eps, G, m, nominal_permit):
    """Explanation of Section 5: minimum-cardinality set J of types, containing the type of the query, such that
    the admissible worlds consistent with the data in which the types in J take the reference factor contain no
    counterexample to the nominal verdict."""
    lows = [know.unit[T][0] for T in TYPES]; highs = [know.unit[T][1] for T in TYPES]
    kb_lo = max(max(lows) - eps, THETA[0]); kb_hi = min(min(highs) + eps, THETA[1])
    for k in range(1, len(TYPES) + 1):
        for J in combinations(TYPES, k):
            if G not in J: continue
            tlo = max([kb_lo] + [know.unit[j][0] for j in J]); thi = min([kb_hi] + [know.unit[j][1] for j in J])
            if tlo > thi: return J
            a, b = (tlo, thi) if G in J else (max(know.unit[G][0], tlo - eps), min(know.unit[G][1], thi + eps))
            if a > b: return J
            if (nominal_permit and _permit_exact(G, m, a, b)) or (not nominal_permit and _block_exact(G, m, a, b)):
                return J
    return None

def explain_undecided(know, G, m, a, b):
    """Remark 1: margin of the estimate divided by the sensitivity of the verdict to the factor."""
    k = ml_estimate(know, G, a, b); i = _idx(k)
    slope = abs(np.gradient(F[(G, m)], KG)[i])
    return abs((1.0 - F[(G, m)][i]) - ETA) / max(slope, 1e-9)


def true_kappa(delta_mat=0.06, delta_cast=0.07, k0=1.0):
    # light boxes are taken from a damping mat (absorbed deviation); castings are oily
    return {"box": k0 + delta_mat, "crate": k0, "casting": k0 - delta_cast}

def sample_query(rng, qp=None):
    qp = qp or QUERY_P
    G = rng.choice(TYPES, p=[qp[t] for t in TYPES])
    return G, int(rng.choice(list(KC_TARGET[G])))

def outcome_probs(gk, fk):
    """Successor distribution of a lift: stable, micro-slip without drop, drop. A slip is detected with
    probability g (micro-slip or drop), and the part is dropped with probability f <= g."""
    return np.array([1.0 - gk, max(gk - fk, 0.0), fk])

def execute(rng, know, kap, G, m, kind="lift", trials=1):
    """Executes `trials` single-lift episodes and records the observed transitions (s, a, s') in D_k."""
    out = rng.multinomial(trials, outcome_probs(val(GT[(G, m)], kap[G]), val(FT[(G, m)], kap[G])))
    slips, drops = int(out[1] + out[2]), int(out[2])
    if know is not None: know.add((G, m, kind), out)
    return slips, drops

COST = dict(H=1.0, T=0.05, F=0.10)
N_OFFER, LEARN_EPS, DROP_PENALTY = 3, 0.1, -10.0
NN_HIDDEN, NN_LR, NN_BATCH, NN_BUFFER, R_SCALE = 16, 0.05, 32, 500, 14.0

class NeuralProposer:
    """Online multilayer perceptron (one hidden layer of tanh units) estimating the reward of a proposal."""
    def __init__(self, rng):
        self.rng = rng
        self.W1 = rng.normal(0.0, 0.5, (5, NN_HIDDEN)); self.b1 = np.zeros(NN_HIDDEN)
        self.W2 = rng.normal(0.0, 0.1, NN_HIDDEN); self.b2 = 0.0
        self.X, self.Y = [], []

    @staticmethod
    def features(G, m):
        return np.array([G == "box", G == "crate", G == "casting", m / 14.0, (m / 14.0) ** 2], dtype=float)

    def predict(self, X):
        return np.tanh(X @ self.W1 + self.b1) @ self.W2 + self.b2

    def choose(self, offers):
        if self.rng.random() < LEARN_EPS: return offers[self.rng.integers(len(offers))]
        v = self.predict(np.array([self.features(*o) for o in offers]))
        best = np.flatnonzero(v == v.max())
        return offers[best[self.rng.integers(len(best))]]

    def update(self, G, m, reward):
        self.X.append(self.features(G, m)); self.Y.append(reward / R_SCALE)
        if len(self.X) > NN_BUFFER: self.X.pop(0); self.Y.pop(0)
        idx = self.rng.integers(len(self.X), size=min(NN_BATCH, len(self.X)))
        X, Y = np.array([self.X[i] for i in idx]), np.array([self.Y[i] for i in idx])
        H = np.tanh(X @ self.W1 + self.b1); err = H @ self.W2 + self.b2 - Y          # squared-error gradient step
        gW2 = H.T @ err / len(Y); gb2 = err.mean()
        dH = np.outer(err, self.W2) * (1 - H ** 2)
        self.W1 -= NN_LR * X.T @ dH / len(Y); self.b1 -= NN_LR * dH.mean(0)
        self.W2 -= NN_LR * gW2; self.b2 -= NN_LR * gb2
K_QUERIES, H_EXP, BUDGET = 400, 20, 10

def run(method, seed, eps=0.10, r=2, delta_cast=0.07, delta_mat=0.06, K=K_QUERIES, ci="kl", schedule=None, learner=True):
    """method in {'ours','closed','conf','plaut'}. Returns per-query logs."""
    rng = np.random.default_rng(seed)
    kap = true_kappa(delta_mat, delta_cast)
    know = Knowledge(ci); know.truth = kap
    cache = {}                                   # decisions taken by the human, reused for identical queries
    loglik = np.zeros_like(KG)                   # pooled likelihood (confidence-based baseline)
    eps_cur, rejected_at, refuted, closed_rejected = eps, None, 0, False
    untied, revisions, explanations = frozenset(), [], []
    log = {k: np.zeros(K) for k in ["unsafe", "asked", "exps", "fallback", "drop", "safe_auto", "executed", "reward", "moved"]}
    proposer = NeuralProposer(np.random.default_rng(10_000 + seed)) if learner else None
    route = np.zeros(K, dtype=int)               # 0 exec-verified, 1 block-verified, 2 test, 3 defer, 4 cache/copy

    def do_exec(t, G, m, autonomous):
        nonlocal loglik
        micro, drop = execute(rng, know, kap, G, m)
        loglik += micro * np.log(Gm[(G, m)]) + (1 - micro) * np.log(1 - Gm[(G, m)])   # slip detected or not
        log["drop"][t] += drop; log["executed"][t] = 1
        if autonomous and not true_safe(G, m, kap): log["unsafe"][t] = 1
        if autonomous and true_safe(G, m, kap): log["safe_auto"][t] = 1

    def ask(t, G, m):
        log["asked"][t] = 1
        ans = true_safe(G, m, kap)               # the human knows the true world
        cache[(G, m)] = ans
        return ans

    def learn(t, G, m):
        if not learner: return
        r_t = DROP_PENALTY if log["drop"][t] > 0 else (m if log["executed"][t] else 0.0)
        log["reward"][t] = r_t; log["moved"][t] = m if (log["executed"][t] and not log["drop"][t]) else 0.0
        proposer.update(G, m, r_t)

    for t in range(K):
        if t > 0: learn(t - 1, *prev)
        qp = schedule(t) if schedule else None
        if learner:
            offers = [sample_query(rng, qp) for _ in range(N_OFFER)]
            G, m = proposer.choose(offers)
        else:
            G, m = sample_query(rng, qp)
        prev = (G, m)

        if method == "plaut":                    # ask-for-help with local generalization
            near = [(abs(m - mm) + (0 if GG == G else 4), GG, mm) for (GG, mm) in cache]
            near = [x for x in near if x[0] <= r]
            if near:
                dmin = min(x[0] for x in near)
                ok = all(cache[(GG, mm)] for d, GG, mm in near if d == dmin)
                route[t] = 4
                if ok: do_exec(t, G, m, autonomous=True)
                else: log["fallback"][t] = 1
            else:
                route[t] = 3
                if ask(t, G, m): do_exec(t, G, m, autonomous=False)
                else: log["fallback"][t] = 1
            continue

        if method == "conf":                     # nominal model + likelihood interval, defer if undecided
            ok = loglik.max() - loglik <= 1.92
            a, b = KG[ok][0], KG[ok][-1]
            if permit(G, m, a, b): route[t] = 0; do_exec(t, G, m, True)
            elif block(G, m, a, b): route[t] = 1; log["fallback"][t] = 1
            elif (G, m) in cache:
                route[t] = 4
                if cache[(G, m)]: do_exec(t, G, m, False)
                else: log["fallback"][t] = 1
            else:
                route[t] = 3
                if ask(t, G, m): do_exec(t, G, m, False)
                else: log["fallback"][t] = 1
            continue

        # ---- ours (eps > 0) and closed-world verification (eps = 0) ----
        e = eps_cur if method == "ours" else 0.0
        R = know.ranges(e)
        if R is None:
            if method == "ours":                 # Assumption 2 refuted: fallback, notify, enlarge eps
                refuted += 1; eps_cur += 0.05; log["fallback"][t] = 1; route[t] = 1
                continue
            closed_rejected = True
            if rejected_at is None: rejected_at = t
        R_nom = None
        if method == "ours":
            R_nom = nominal_ranges(know, untied, e)
            if R_nom is None:                    # nominal class rejected: notify and revise (Remark 3)
                if rejected_at is None: rejected_at = t
                new = revise(know, untied, e)
                if new is not None:
                    untied = new; revisions.append((t, tuple(sorted(new))))
                    R_nom = nominal_ranges(know, untied, e)
        if method == "closed" and closed_rejected:
            if (G, m) in cache:
                route[t] = 4
                if cache[(G, m)]: do_exec(t, G, m, False)
                else: log["fallback"][t] = 1
            else:
                route[t] = 3
                if ask(t, G, m): do_exec(t, G, m, False)
                else: log["fallback"][t] = 1
            continue
        a, b = R[G]
        if permit(G, m, a, b): route[t] = 0; do_exec(t, G, m, True); continue
        if block(G, m, a, b):  route[t] = 1; log["fallback"][t] = 1; continue
        if (G, m) in cache:
            route[t] = 4
            if cache[(G, m)]: do_exec(t, G, m, False)
            else: log["fallback"][t] = 1
            continue
        units = [G] if e > 0 else TYPES
        cands = [(GG, mm, "probe") for (GG, mm, kd) in PAIRS if kd == "probe" and GG in units]
        cands += [(GG, mm, "lift") for (GG, mm) in QUERIES if GG in units and permit(GG, mm, *R[GG])]
        if cands:                                # safely testable: fallback now, certified experiments
            route[t] = 2; log["fallback"][t] = 1
            for _ in range(BUDGET):
                R = know.ranges(e)
                if R is None: break
                a, b = R[G]
                if permit(G, m, a, b) or block(G, m, a, b): break
                khat = ml_estimate(know, G, a, b)
                kalt = opposite_world(G, m, a, b, khat)
                if kalt is None: break
                # rule (9): certified behavior maximizing the divergence between the estimate and the closest
                # world of the opposite verdict (one single-lift episode; the KL is per episode)
                best = max(cands, key=lambda c: kl_cat(lift_dist(c[0], c[1], khat), lift_dist(c[0], c[1], kalt)))
                execute(rng, know, kap, best[0], best[1], best[2], trials=H_EXP)
                log["exps"][t] += H_EXP                 # experiment episodes (single lifts)
        else:                                    # not safely testable: defer with explanation
            route[t] = 3
            if method == "ours":
                nom = None
                if R_nom is not None:
                    na, nb = R_nom[G]
                    nom = "permit" if _permit_exact(G, m, na, nb) else ("block" if _block_exact(G, m, na, nb) else None)
                if nom is not None:              # model-dependent query: structural explanation J
                    explanations.append((t, G, m, "model-dependent", nom, explain_model_dependent(know, e, G, m, nom == "permit")))
                else:                            # undecided query: parametric explanation
                    explanations.append((t, G, m, "undecided", None, round(explain_undecided(know, G, m, a, b), 3)))
            if ask(t, G, m): do_exec(t, G, m, False)
            else: log["fallback"][t] = 1

    if K > 0: learn(K - 1, *prev)
    log["route"] = route
    log["rejected_at"] = rejected_at
    log["refuted"] = refuted
    log["revisions"] = revisions
    log["explanations"] = explanations
    log["cov_fail"] = know.cov_fail
    return log

def summarize(logs):
    u = np.array([l["unsafe"].sum() for l in logs]); h = np.array([l["asked"].sum() for l in logs])
    x = np.array([l["exps"].sum() for l in logs]); f = np.array([l["fallback"].sum() for l in logs])
    c = COST["H"] * h + COST["T"] * x + COST["F"] * f
    sa = np.array([l["safe_auto"].sum() for l in logs])
    mv = np.array([l["moved"].sum() for l in logs]); dr = np.array([l["drop"].sum() for l in logs])
    return dict(safe=sa.mean(), moved=mv.mean(), drops=dr.mean(), unsafe=u.mean(), unsafe_se=u.std(ddof=1) / np.sqrt(len(u)), asked=h.mean(),
                asked_se=h.std(ddof=1) / np.sqrt(len(h)), exps=x.mean(), fallback=f.mean(), cost=c.mean())

plt.rcParams.update({"font.size": 11, "axes.labelsize": 11,
                     "axes.spines.top": False, "axes.spines.right": False,
                     "axes.grid": True, "grid.alpha": 0.25, "legend.frameon": False})
C_OURS, C_PL, C_CL, C_CF, C_GREY = "#3E8E5E", "#C9831C", "#22304D", "#C0453A", "#5F6368"

def save(fig, name):
    fig.tight_layout(); fig.savefig(f"{OUT}/{name}.pdf"); fig.savefig(f"{OUT}/{name}.png", dpi=200)
    plt.show()

def bar_labels(ax, bars, fmt="{:.1f}"):
    for b in bars:
        ax.text(b.get_x() + b.get_width() / 2, b.get_height(), fmt.format(b.get_height()),
                ha="center", va="bottom", fontsize=10, fontweight="bold")

METHODS = [("ours",   dict(eps=0.10), "Ours",                   C_OURS),
           ("plaut",  dict(r=2),      "Plaut et al.\n(r = 2)",  C_PL),
           ("plaut",  dict(r=0),      "Plaut et al.\n(r = 0)",  "#E3B66B"),
           ("closed", dict(),         "Closed-world\n(ε = 0)",  C_CL),
           ("conf",   dict(),         "Confidence-\nbased",     C_CF)]


print("Main scenario ...")
SEEDS = range(40)
MAIN = {lab: summarize([run(k, s, **kw) for s in SEEDS]) for k, kw, lab, c in METHODS}

fig, axs = plt.subplots(1, 2, figsize=(10.5, 4.0))
labs = [lab for _, _, lab, _ in METHODS]; cols = [c for _, _, _, c in METHODS]
for ax, key, ylab, fmt in [(axs[0], "unsafe", "Unsafe actions executed\nwithout the human (per run)", "{:.1f}"),
                           (axs[1], "safe", "Safe actions executed\nwithout the human (per run)", "{:.0f}")]:
    b = ax.bar(range(len(labs)), [MAIN[l][key] for l in labs], color=cols, width=0.7); bar_labels(ax, b, fmt)
    ax.set_xticks(range(len(labs)))
    ax.set_xticklabels(["Ours", "Plaut et al.\nr = 2", "Plaut et al.\nr = 0", "Closed-\nworld", "Confidence-\nbased"], fontsize=9)
    ax.set_ylabel(ylab); ax.grid(axis="x", visible=False)
for ax, letter in zip(axs, "ab"):
    ax.text(-0.02, 1.04, f"({letter})", transform=ax.transAxes, fontsize=11, fontweight="bold")
save(fig, "fig1_safety_and_deferrals")


print("Trade-off sweep ...")
SW = range(20); r_list = [0, 1, 2, 3, 4, 6]
P_pl = [summarize([run("plaut", s, r=r) for s in SW]) for r in r_list]
fig, ax = plt.subplots(figsize=(6.4, 4.4))
ax.plot([p["asked"] for p in P_pl], [p["unsafe"] for p in P_pl], "o-", color=C_PL, lw=2.5, ms=7)
OFF = {0: (8, 2), 1: (8, 4), 2: (-12, -16), 3: (8, 4), 4: (-36, -4), 6: (-36, -4)}
for p, rr in zip(P_pl, r_list):
    ax.annotate(f"r = {rr}", (p["asked"], p["unsafe"]), textcoords="offset points", xytext=OFF[rr], fontsize=9, color=C_PL)
ax.text(12.5, 0.8 * max(p["unsafe"] for p in P_pl), "Plaut et al.\n(asks within radius r)", color=C_PL, fontsize=10)
pts = [("Closed-world (ε = 0)", MAIN["Closed-world\n(ε = 0)"], C_CL, "D"), ("Confidence-based", MAIN["Confidence-\nbased"], C_CF, "^"),
       ("Ours", MAIN["Ours"], C_OURS, "*")]
for name, p, col, mk in pts:
    ax.plot(p["asked"], p["unsafe"], mk, color=col, ms=18 if mk == "*" else 10)
    off = {"*": (10, 8), "^": (-12, -4), "D": (10, 6)}[mk]
    ax.annotate(name, (p["asked"], p["unsafe"]), textcoords="offset points", xytext=off, ha="right" if mk == "^" else "left",
                fontsize=10, color=col, fontweight="bold" if mk == "*" else "normal")
ax.set_xlim(-0.5, 25); ax.set_ylim(-1, max(p["unsafe"] for p in P_pl) + 4)
ax.set_xlabel("Decisions deferred to the human (per run)"); ax.set_ylabel("Unsafe actions executed\nwithout the human (per run)")
save(fig, "fig2_tradeoff")


print("Lifelong scenario ...")
T1, KL, WIN = 300, 900, 50
SCHED = lambda t: {"box": 0.55, "crate": 0.45, "casting": 0.0} if t < T1 else {"box": 0.35, "crate": 0.30, "casting": 0.35}
LL_METHODS = [("ours", dict(), "Ours", C_OURS, "-"), ("plaut", dict(r=2), "Plaut et al. (r = 2)", C_PL, "-"),
              ("plaut", dict(r=0), "Plaut et al. (r = 0)", "#E3B66B", "--"), ("closed", dict(), "Closed-world (ε = 0)", C_CL, "-"),
              ("conf", dict(), "Confidence-based", C_CF, "-")]
SL = range(30)
LL = {lab: [run(k, s, K=KL, schedule=SCHED, **kw) for s in SL] for k, kw, lab, c, ls in LL_METHODS}

def per_window(logs, key):
    return np.array([l[key] for l in logs]).mean(0).reshape(-1, WIN).sum(1) * (100 / WIN)

def lifelong_axes(ax, title, ylabel):
    ax.axvspan(0, T1, color="#E6ECF3", alpha=0.6, lw=0); ax.axvline(T1, color=C_GREY, ls="--", lw=1)
    ax.text(T1 / 2, 1.02, "boxes and crates", transform=ax.get_xaxis_transform(), ha="center", fontsize=9, color=C_GREY)
    ax.text((T1 + KL) / 2, 1.02, "castings are introduced", transform=ax.get_xaxis_transform(), ha="center", fontsize=9, color=C_GREY)
    ax.set_xlim(0, KL); ax.set_xlabel("Proposed actions over the lifetime of the robot"); ax.set_ylabel(ylabel)

xw = np.arange(WIN / 2, KL, WIN)

fig, ax = plt.subplots(figsize=(7.4, 4.0))
ends = []
for k, kw, lab, col, ls in LL_METHODS:
    if lab == "Plaut et al. (r = 0)": continue
    y = per_window(LL[lab], "unsafe"); ax.plot(xw, y, ls, color=col, lw=2.5, marker="o", ms=4)
    ends.append([y[-1], lab, col])
ends.sort(); gap = 0.1 * max(e[0] for e in ends + [[1, "", ""]])
for i in range(1, len(ends)):
    ends[i][0] = max(ends[i][0], ends[i - 1][0] + gap)
for yv, lab, col in ends:
    ax.text(KL + 8, yv, lab, color=col, fontsize=9, va="center")
lifelong_axes(ax, None, "Unsafe actions executed without the\nhuman (per 100 proposed actions)")
ax.set_xlim(0, KL + 5); fig.subplots_adjust(right=0.78)
save(fig, "fig3_lifelong_unsafe")


fig, ax = plt.subplots(figsize=(7.4, 4.0))
for k, kw, lab, col, ls in LL_METHODS[1:] + LL_METHODS[:1]:
    y = per_window(LL[lab], "asked")
    ax.plot(xw, y, ls, color=col, lw=3.5 if lab == "Ours" else 2.2, marker="o", ms=4, label=lab, zorder=5 if lab == "Ours" else 2)
lifelong_axes(ax, None, "Decisions deferred to the human\n(per 100 proposed actions)")
h, l = ax.get_legend_handles_labels(); order = [l.index("Ours")] + [i for i in range(len(l)) if l[i] != "Ours"]
ax.legend([h[i] for i in order], [l[i] for i in order], fontsize=9, loc="upper right")
save(fig, "fig4_lifelong_deferrals")

# Figure 5: the robot detects that its model is wrong and revises it
fig, ax = plt.subplots(figsize=(7.4, 4.0))
for lab, col in [("Ours", C_OURS), ("Closed-world (ε = 0)", C_CL)]:
    times = np.array([l["rejected_at"] if l["rejected_at"] is not None else np.inf for l in LL[lab]])
    frac = [np.mean(times <= t) for t in range(KL)]
    ax.plot(range(KL), frac, color=col, lw=2.5); ax.text(KL + 8, frac[-1], lab, color=col, fontsize=9, va="center")
for lab, col in [("Confidence-based", C_CF), ("Plaut et al.", C_PL)]:
    ax.plot(range(KL), np.zeros(KL), color=col, lw=2.5, ls=":")
ax.text(KL + 8, -0.04, "Confidence-based,\nPlaut et al.\n(no detection)", color=C_GREY, fontsize=9, va="center")
lifelong_axes(ax, None, "Fraction of runs in which the robot\nhas detected that its model is wrong")
ax.set_ylim(-0.08, 1.05)
save(fig, "fig5_lifelong_model_revision")


print("Polynomial approximation of the physics ...")
FAMS = [("linear", "Linear"), ("bernstein-6", "Bernstein 6"), ("bernstein-15", "Bernstein 15"),
        ("bernstein-30", "Bernstein 30"), ("bernstein-60", "Bernstein 60"), ("chebyshev-15", "Chebyshev 15")]
SF = range(15); famres = {}
for fam, _ in FAMS:
    build_family(fam, "physics")
    famres[fam] = dict(err=np.mean([np.abs(Gm[q] - GT[q]).max() for q in QUERIES]),
                       **{m: summarize([run(m, sd) for sd in SF]) for m in ["ours", "closed", "conf"]})
build_default()
fig, ax = plt.subplots(figsize=(7.4, 4.2))
xx = np.arange(len(FAMS)); wbar = 0.26
for j, (m, lab, col) in enumerate([("ours", "Ours", C_OURS), ("closed", "Closed-world (ε = 0)", C_CL), ("conf", "Confidence-based", C_CF)]):
    y = [famres[f][m]["unsafe"] for f, _ in FAMS]
    ax.bar(xx + (j - 1) * wbar, y, wbar, color=col, label=lab)
    if m == "ours":
        for i, v in enumerate(y):
            if v == 0: ax.text(xx[i] - wbar, 0.3, "0", ha="center", va="bottom", fontsize=9, color=col, fontweight="bold")
ax.set_xticks(xx); ax.set_xticklabels([f"{n}\n(error {famres[f]['err']:.2f})" for f, n in FAMS], fontsize=8.5)
ax.set_ylabel("Unsafe actions executed\nwithout the human (per run)"); ax.grid(axis="x", visible=False)
ax.legend(fontsize=9, loc="upper right")
save(fig, "fig6_polynomial_approximation")


print("Certification with Bernstein and Chebyshev polynomials ...")
WIDTHS = [(0.03, 0.06), (0.015, 0.03), (0.008, 0.015)]
CFAMS = [("bernstein-15", "Bernstein 15", "#A9D3B9", "-"), ("bernstein-30", "Bernstein 30", "#6BAF87", "-"),
         ("bernstein-60", "Bernstein 60", C_OURS, "-"), ("chebyshev-6", "Chebyshev 6", "#F0C27B", "--"),
         ("chebyshev-10", "Chebyshev 10", C_PL, "--"), ("chebyshev-15", "Chebyshev 15", "#8C5A12", "--")]
SC = range(15); cert = {}
CERT["mode"] = "monotone"
for wd, wg in WIDTHS:
    for fam, name, col, ls in CFAMS:
        inval = build_family(fam, "same", wd, wg)
        CERT["certs"] = CERT["unsound"] = 0
        L = [run("ours", sd) for sd in SC]
        cert[(wd, fam)] = dict(unsound=100 * CERT["unsound"] / max(CERT["certs"], 1),
                               cov=100 * np.mean([l["cov_fail"] for l in L]),
                               unsafe=summarize(L)["unsafe"], invalid=inval)
CERT["mode"] = "exact"; build_default()
fig, axs = plt.subplots(1, 3, figsize=(14, 4.2))
xw = [w for w, _ in WIDTHS]
for ax, key, ylab in [(axs[0], "unsound", "Unsound safety certificates\n(% of issued certificates)"),
                      (axs[1], "cov", "Runs in which the true parameter\nleaves its interval (%)"),
                      (axs[2], "unsafe", "Unsafe actions executed\nwithout the human (per run)")]:
    for fam, name, col, ls in CFAMS:
        ax.plot(range(len(xw)), [cert[(w, fam)][key] for w in xw], ls, marker="o", color=col, lw=2.5, label=name)
    ax.set_xticks(range(len(xw))); ax.set_xticklabels([f"{w:.3f}" for w in xw])
    ax.set_xlabel("Width of the physical slip transition"); ax.set_ylabel(ylab)
for ax, letter in zip(axs, "abc"):
    ax.text(-0.02, 1.04, f"({letter})", transform=ax.transAxes, fontsize=11, fontweight="bold")
axs[0].legend(fontsize=8.5, loc="upper left")
save(fig, "fig7_certification")


print(f"\nMain scenario, mean per run of {K_QUERIES} proposed actions")
print(f"{'method':28s} {'unsafe':>7s} {'deferrals':>9s} {'safe auto':>9s} {'experiments':>11s}")
for k, kw, lab, c in METHODS:
    print(f"{lab.replace(chr(10), ' '):28s} {MAIN[lab]['unsafe']:7.2f} {MAIN[lab]['asked']:9.2f} {MAIN[lab]['safe']:9.1f} {MAIN[lab]['exps']:11.1f}")
print("\nLifelong revision of the nominal class (ours, lifelong scenario)")
import collections
revs = [l["revisions"] for l in LL["Ours"]]
print(f"  runs with at least one revision: {np.mean([len(r) > 0 for r in revs]):.2f}")
print("  final structure of the nominal class (types with their own factor):",
      dict(collections.Counter(r[-1][1] if r else () for r in revs)))
print("\nExplanations attached to deferrals (ours, main scenario, 20 runs)")
EX = [x for sd in range(20) for x in run("ours", sd)["explanations"]]
print("  model-dependent:", sum(x[3] == "model-dependent" for x in EX), " undecided:", sum(x[3] == "undecided" for x in EX))
print("  structural explanations J:", dict(collections.Counter(x[5] for x in EX if x[3] == "model-dependent")))
for x in [x for x in EX if x[3] == "model-dependent"][:2]:
    print(f"  e.g. lifting the {x[2]} kg {x[1]} is {'safe' if x[4] == 'permit' else 'unsafe'} provided that the factors of "
          + " and ".join(x[5]) + " coincide with the common reference")
print("\nCertification relying on Assumption 1 (environment = model polynomial)")
print(f"{'width':>6s} {'family':13s} {'invalid curves':>14s} {'unsound cert %':>14s} {'coverage fail %':>15s} {'unsafe':>7s}")
for wd, wg in WIDTHS:
    for fam, name, col, ls in CFAMS:
        c = cert[(wd, fam)]
        print(f"{wd:6.3f} {name:13s} {c['invalid']:11d}/23 {c['unsound']:14.2f} {c['cov']:15.1f} {c['unsafe']:7.2f}")
print(f"\nTotal time {time.time() - T0:.0f}s. Figures saved in ./{OUT}/")